# Aula 2 · Análise Exploratória do Olist (EDA)

Antes de treinar qualquer modelo, o bom engenheiro de dados **olha os dados**.
Esta é a etapa de **análise exploratória** (EDA): entender o que temos, como o
alvo se distribui e quais padrões já saltam aos olhos.

> Roteiro: visão geral → distribuição do alvo → cobertura e viés → geografia,
> categorias, entrega → a relação que mais importa (atraso × satisfação).

## 1. Carregar os dados

In [ ]:
import urllib.request, pathlib
import pandas as pd
import matplotlib.pyplot as plt

URL = "https://github.com/fenakamuta/poliusppro-data-engineering/releases/download/aula1-olist-2026-v1/olist.parquet"
PATH = pathlib.Path("olist.parquet")
if not PATH.exists():
    print("Baixando..."); urllib.request.urlretrieve(URL, PATH)

df = pd.read_parquet(PATH)
print(f"{len(df):,} pedidos · {df.shape[1]} colunas")
df.head()

## 2. Visão geral das colunas

Quais informações temos sobre cada pedido?

In [ ]:
df.dtypes.to_frame("tipo")

## 3. A distribuição do ALVO (a nota)

O que queremos prever é a satisfação do cliente, medida pela **nota de 1 a 5 estrelas**.
Antes de tudo: como ela se distribui?

In [ ]:
contagem = df["review_score"].value_counts().sort_index()
ax = contagem.plot(kind="bar", color="#7048e8", figsize=(8, 4), edgecolor="white")
ax.set_title("Distribuição das notas (estrelas)", fontweight="bold")
ax.set_xlabel("Estrelas"); ax.set_ylabel("Pedidos")
for i, v in enumerate(contagem):
    ax.text(i, v + 800, f"{v/len(df)*100:.0f}%", ha="center", fontweight="bold")
plt.tight_layout(); plt.show()

**Observação:** a maioria esmagadora é **5 estrelas** (~59%). As notas baixas
(1-2) somam ~13%. Isso já avisa: a base é **desbalanceada** — tem muito mais cliente
satisfeito do que insatisfeito. Guarde isso, vai importar na hora de avaliar o modelo.

## 4. O alvo binário: satisfeito ou não

Para o modelo, simplificamos: **nota ≥ 4 = satisfeito (1)**, abaixo disso = insatisfeito (0).

In [ ]:
df["satisfeito"] = (df["review_score"] >= 4).astype(int)
dist = df["satisfeito"].value_counts()
pct_pos = df["satisfeito"].mean() * 100

ax = dist.plot(kind="bar", color=["#e03131", "#2f9e44"], figsize=(6, 4), edgecolor="white")
ax.set_xticklabels(["Insatisfeito (0)", "Satisfeito (1)"], rotation=0)
ax.set_title("Alvo binário", fontweight="bold"); ax.set_ylabel("Pedidos")
plt.tight_layout(); plt.show()

print(f"Satisfeitos: {pct_pos:.1f}%   |   Insatisfeitos: {100-pct_pos:.1f}%")

**A armadilha do desbalanceamento:** como ~78% são satisfeitos, um modelo
"preguiçoso" que chuta **sempre satisfeito** já acerta 78%. Esse é o **baseline
ingênuo** — qualquer modelo de verdade tem que superá-lo. Por isso, mais à frente,
não olhamos só a acurácia (olhamos matriz de confusão, precisão, recall).

## 5. Quem escreve comentário? (cobertura)

Nem todo cliente deixa um texto. Quantos deixam?

In [ ]:
tem_texto = df["review_comment_message"].fillna("").str.len() > 0
cob = pd.Series({"Com comentário": tem_texto.sum(), "Sem comentário": (~tem_texto).sum()})
ax = cob.plot(kind="bar", color=["#2f9e44", "#adb5bd"], figsize=(6, 4), edgecolor="white")
ax.set_title("Cobertura do texto", fontweight="bold"); ax.set_ylabel("Pedidos")
ax.set_xticklabels(cob.index, rotation=0)
for i, v in enumerate(cob):
    ax.text(i, v + 800, f"{v/len(df)*100:.0f}%", ha="center", fontweight="bold")
plt.tight_layout(); plt.show()

Só ~40% escrevem. Um modelo que dependesse **só do texto** seria **cego** para
a maioria — daí a importância de usar também os dados de tabela (preço, prazo...).

## 6. Viés de seleção: quem escreve é diferente?

Quem se dá ao trabalho de escrever tende a ser mais... bravo?

In [ ]:
df["tem_texto"] = tem_texto
comp = df.groupby("tem_texto")["review_score"].mean()
print(f"Score médio — quem ESCREVEU comentário: {comp[True]:.2f}")
print(f"Score médio — quem só deu estrela:      {comp[False]:.2f}")

**Viés de seleção:** quem escreve tem nota média mais baixa. A amostra de texto
é "torta" — puxada para os insatisfeitos. É um conceito central: os dados que você
coleta nem sempre representam todo mundo.

## 7. Onde estão os pedidos (geografia)

In [ ]:
top_uf = df["customer_state"].value_counts().head(10)
ax = top_uf.plot(kind="bar", color="#1971c2", figsize=(9, 4), edgecolor="white")
ax.set_title("Top 10 estados por nº de pedidos", fontweight="bold"); ax.set_ylabel("Pedidos")
plt.tight_layout(); plt.show()
print(f"SP sozinho: {top_uf.iloc[0]/len(df)*100:.0f}% dos pedidos")

## 8. Categorias mais vendidas

In [ ]:
top_cat = df["product_category"].value_counts().head(10)
ax = top_cat.plot(kind="barh", color="#2f9e44", figsize=(9, 5))
ax.set_title("Top 10 categorias", fontweight="bold"); ax.invert_yaxis()
plt.tight_layout(); plt.show()

## 9. Tempo de entrega

In [ ]:
ax = df["delivery_days"].clip(upper=60).plot(kind="hist", bins=40, color="#7048e8",
                                              figsize=(9, 4), edgecolor="white")
ax.set_title("Distribuição dos dias de entrega (cap em 60)", fontweight="bold")
ax.set_xlabel("Dias até entregar"); plt.tight_layout(); plt.show()
print(f"Mediana: {df['delivery_days'].median():.0f} dias  ·  "
      f"% atrasados: {df['delivered_late'].mean()*100:.0f}%")

## 10. A relação que mais importa: atraso × satisfação

Esta é a descoberta de ouro da EDA — e o que mais vai pesar no modelo.

In [ ]:
rel = df[df["delivery_days"].notna()].groupby("delivered_late").agg(
    pedidos=("satisfeito", "size"),
    score_medio=("review_score", "mean"),
    pct_satisfeito=("satisfeito", "mean"),
).round(2)
rel["pct_satisfeito"] = (rel["pct_satisfeito"] * 100).round(0)
print(rel)

ax = rel["score_medio"].plot(kind="bar", color=["#2f9e44", "#e03131"], figsize=(6, 4), edgecolor="white")
ax.set_xticklabels(["No prazo", "Atrasado"], rotation=0)
ax.set_title("Nota média: no prazo vs atrasado", fontweight="bold"); ax.set_ylabel("Nota média")
plt.tight_layout(); plt.show()

**Atraso destrói a satisfação:** no prazo a nota média é ~4.3; atrasado despenca
para ~2.3. É de longe o sinal mais forte — e funciona mesmo para quem não escreveu
comentário. Por isso o modelo combinado (texto + tabela) consegue prever todo mundo.

## 11. Preço e frete

In [ ]:
print(df[["price", "freight_value"]].describe().round(2))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
df["price"].clip(upper=500).plot(kind="hist", bins=40, ax=axes[0], color="#1971c2", edgecolor="white")
axes[0].set_title("Preço (cap R$500)")
df["freight_value"].clip(upper=100).plot(kind="hist", bins=40, ax=axes[1], color="#e8590c", edgecolor="white")
axes[1].set_title("Frete (cap R$100)")
plt.tight_layout(); plt.show()

## Recapitulando

| Achado | Por que importa |
|--------|-----------------|
| Alvo desbalanceado (~78% satisfeitos) | Acurácia sozinha engana; baseline ingênuo é 78% |
| Só ~40% escrevem comentário | Modelo só de texto é cego para a maioria → use a tabela |
| Quem escreve é mais bravo | Viés de seleção: amostra de texto não representa todos |
| Atraso derruba a nota (4.3 → 2.3) | Sinal mais forte; prevê até sem texto |

Agora que **entendemos os dados**, vamos representá-los e modelar:
- [`01-como-texto-vira-numero-tfidf.ipynb`](./01-como-texto-vira-numero-tfidf.ipynb)
- [`02-do-texto-ao-modelo.ipynb`](./02-do-texto-ao-modelo.ipynb)